In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import numpy as np
from matplotlib.colors import ListedColormap
import random
import colorsys

In [2]:

def generate_extended_palette(n_colors):
    # Generate a large palette by cycling through seaborn palettes with different hues
    base_palette = sns.color_palette("tab20", min(n_colors, 20))
    extended_palette = base_palette
    while len(extended_palette) < n_colors:
        additional_palette = sns.color_palette("hsv", n_colors)
        extended_palette.extend(additional_palette[: n_colors - len(extended_palette)])

    # Remove grey-like colors from the palette
    def is_grey(color):
        r, g, b = color
        return abs(r - g) < 0.1 and abs(g - b) < 0.1 and abs(r - b) < 0.1

    extended_palette = [color for color in extended_palette if not is_grey(color)]

    # Ensure the palette has enough colors
    while len(extended_palette) < n_colors:
        additional_palette = sns.color_palette("husl", n_colors)
        extended_palette.extend(additional_palette[: n_colors - len(extended_palette)])
        extended_palette = [color for color in extended_palette if not is_grey(color)]

    return extended_palette[:n_colors]

In [8]:
barcode_file = "/grid/siepel/home/staklins/projects/crispr_barcode/data/heritable_silencing_01_11_16_25_uniform_50cells_50sites_0.0025mut_10-6mig_data_from_8_22_24/2588/2588_indel_character_matrix.tsv"
outfile = barcode_file.replace(".tsv", ".pdf")

barcode_df = pd.read_csv(barcode_file, sep="\t", index_col=0)

# Merge the barcode and tissue dataframes
num_barcode_sites = barcode_df.shape[1]

# Prepare unique colors for each integer value
unique_values = np.unique(
    [
        val
        for val in barcode_df.iloc[:, :].values.flatten()
        if val != -1 and val != 0
    ]
)
value_to_index = {val: i for i, val in enumerate(sorted(unique_values))}

# Adjust colors: 0 as white, -1 as grey, others with an extended color palette
extended_palette = generate_extended_palette(len(unique_values))
values = sorted(np.unique(barcode_df.values))
if 0 in values:
    extended_palette = ["white"] + extended_palette
    value_to_index = {key: value+1 for key, value in value_to_index.items()}
    value_to_index[0] = 0
if -1 in values:
    extended_palette = ["grey"] + extended_palette
    value_to_index = {key: value+1 for key, value in value_to_index.items()}
    value_to_index[-1] = 0
index_to_color = [extended_palette[i] for i, val in enumerate(values)]

# Create a custom colormap for the dashed line fill
cmap = ListedColormap(index_to_color)
cmap.set_bad(color="black", alpha=0.0)  # Set NaN values to be transparent

indexed_data = barcode_df.iloc[:, :].replace(value_to_index)

plt.figure(figsize=(6, 6))

ax = sns.heatmap(
    indexed_data,
    annot=False,
    fmt="d",
    cmap=cmap,
    cbar=False,
    linewidths=0.0,
    linecolor="black",
)


plt.yticks(np.arange(len(barcode_df.index)) + 0.5, barcode_df.index, rotation=0)
plt.xlabel(f"Barcode sites")
plt.ylabel(f"Cells")
plt.title("")

fs = 18
ax.tick_params(axis="x", labelsize=fs, rotation=45)
ax.tick_params(axis="y", labelsize=fs)
ax.set_title(ax.get_title(), fontsize=fs)
ax.set_xlabel(ax.get_xlabel(), fontsize=fs)
ax.set_ylabel(ax.get_ylabel(), fontsize=fs)
ax.set_yticklabels([])
ax.set_xticklabels([])
ax.tick_params(axis="both", length=0)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)
    spine.set_color("black")

plt.tight_layout()
plt.savefig(outfile)
# plt.show()
plt.close()

